원본 데이터 소스 : https://github.com/e9t/nsmc

In [1]:
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import numpy

load_dotenv(find_dotenv())

True

In [2]:
from openai import OpenAI
client = OpenAI()

In [3]:
DATA_TRAIN_PATH = "https://raw.githubusercontent.com/sollab-source/sollab-source.github.io/main/data/ratings_train.txt"
DATA_TEST_PATH = "https://raw.githubusercontent.com/sollab-source/sollab-source.github.io/main/data/ratings_test.txt"

In [4]:
train_df = pd.read_csv(DATA_TRAIN_PATH, delimiter="\t")
test_df = pd.read_csv(DATA_TEST_PATH, delimiter="\t")

In [7]:
train_df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [8]:
test_df.head()

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [ ]:
print(train_df.shape)
print(test_df.shape)

(150000, 3)
(50000, 3)


In [ ]:
n_train_samples = 500
n_test_samples = 100

# 긍정, 부정 리뷰추출(학습 데이터)
train_positive = train_df[train_df.label == 1]
train_negetive = train_df[train_df.label == 0]

# 긍정, 부정 리뷰추출(시험 데이터)
test_positive = test_df[test_df.label == 1]
test_negetive = test_df[test_df.label == 0]

# 추출된 데이터에서 무작위로 학습 500/시험 100개 추출
train_positive_sample = train_positive.sample(n_train_samples, random_state=42)
train_negetive_sample = train_negetive.sample(n_train_samples, random_state=42)

test_positive_sample = test_positive.sample(n_test_samples, random_state=42)
test_negetive_sample = test_negetive.sample(n_test_samples, random_state=42)

# 학습 데이터 결합
train_sample = pd.concat(
    [train_positive_sample, train_negetive_sample], ignore_index=True
)
train_sample.to_csv("./data/train_sample.csv", index=False)

# 시험 데이터 결합
test_sample = 

In [5]:
# 임베딩 생성(사이킷런 등 라이브러리 이용 => 직접 가능)
# 임베딩 생성(openai)


def get_embedding(text, batch_size=256):  # batch_size : 한꺼번에 같이 임베딩할 크기
    vectors = []
    for i in range(0, len(text), batch_size):
        batch = text[i : i + batch_size]

        response = client.embeddings.create(input=batch, model="text-embedding-3-small")

        vectors.extend([d.embedding for d in response.data])

    return numpy.array(vectors, dtype=numpy.float32)

In [ ]:
# document 내용
train_texts = train_sample.document.to_list()
test_texts = test_sample.document.to_list()

X_train = get_embedding(train_texts)
X_test = get_embedding(test_texts)

# 받은 임베딩을 numpy 형태로 저장
numpy.save("./data/X_train.npy", X_train)
numpy.save("./data/X_test.npy", X_test)

In [ ]:
# npy 파일 불러오기

X_train_embedding = numpy.load("./data/X_train.npy")
X_test_embedding = numpy.load("./data/X_test.npy")

In [ ]:
# 학습 데이터 변수 => X_train
# 시험 데이터 변수 => X_test
X_train = X_train_embedding
y_train = train_sample.label.values

X_test = X_test_embedding
y_test = test_sample.label.values

X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
# 훈련 : 머신러닝 라이브러리 사용(sklearn)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=2000)

# 학습
lr.fit(X_train, y_train)

# 시험
preds = lr.predict(X_test)

# 정확도 측정
print(accuracy_score(y_test,preds))

In [ ]:
msg = "이 영화 ㅈㄴ재밌음, 배우들 연기 지리고 스토리라인 탄탄함"

# 긍정 or 부정
# 사용자 메세지 embedding
msg_emb = get_embedding([msg], 1)

# 모델에게 예측시키기
pred_label = lr.predict(msg_emb)[0]
pred_proba = lr.predict_proba(msg_emb)[0]
confidence = float(pred_proba[pred_label])

sentiment = "긍정" if pred_label == 1 else "부정"
confidence = pred_proba[pred_label]

print(sentiment, confidence)

In [ ]:
# 비슷한 리뷰 찾기(코사인 유사도)
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity([msg_emb], X_train)[0]

# 유사한 리뷰 5개 추출
top_idx = similarity.argsort()[-5:][::-1]

for i in top_idx:
    doc = train_sample.iloc[i]["document"]
    label = train_sample.iloc[i]["label"]
    sim = similarity[i]
    print(f"sim={sim:.3f} {label} | {doc}")